# Caso Pratico EMFI 1 – Parte B
**Autori:** Leonardo Pratelli, Sara Albotica – Università di Pisa  
**Docente:** Prof. Fulvio Corsi

---

## Parte B – Stima dei Beta, Frontiera Efficiente, CAPM

Questo notebook implementa i **Punti 8 e 9** della traccia:
- **Punto 8**: Stima OLS dei beta dei 10 titoli rispetto al portafoglio di mercato (S&P 500). Verifica della significatività statistica di beta e alpha di Jensen.
- **Punto 9**: Ricostruzione della matrice varianza-covarianza tramite il **Single Index Model (SIM)** e dei rendimenti attesi tramite il **CAPM**. Costruzione della frontiera efficiente e confronto con quella empirica della Parte A.

---
## Blocco 0 – Import e configurazione

In [ ]:
# =============================================================================
# IMPORT
# =============================================================================
# Le stesse librerie della Parte A, più scipy.stats per la statistica OLS.
# scipy.stats.t: distribuzione t di Student per i p-value dei coefficienti.

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.optimize import minimize

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.size'] = 11

# =============================================================================
# CONFIGURAZIONE ASSET (identica alla Parte A per coerenza)
# =============================================================================
# Stessa struttura 3-3-3-1 della Parte A.
# I 5 titoli selezionati nella Parte A vengono riutilizzati per il confronto
# tra frontiera empirica e frontiera SIM al Punto 9.

TICKERS = {
    'Tech':       ['AAPL', 'MSFT', 'NVDA'],
    'Healthcare': ['JNJ',  'PFE',  'MRK'],
    'Energy':     ['XOM',  'CVX',  'BP'],
    'Index':      ['^GSPC'],
}
ALL_TICKERS = [t for group in TICKERS.values() for t in group]

# 5 titoli selezionati nella Parte A per la frontiera efficiente
SELECTED = ['NVDA', 'MSFT', 'JNJ', 'MRK', 'XOM']

# Tasso risk-free: stesso valore usato nella Parte A
rf_annual = 0.02
ann       = 12
rf        = rf_annual / ann     # tasso mensile

START = '2015-01-01'
END   = '2025-01-01'

print('Titoli:', ALL_TICKERS)
print(f'Risk-free: {rf_annual*100:.1f}% annuo | Periodo: {START} → {END}')

---
## Blocco 1 – Download dati e rendimenti

Stesso procedimento della Parte A (notebook autonomo, non dipende dai dati di notebook_A).
I rendimenti logaritmici mensili vengono ricalcolati da zero.

In [ ]:
# Download prezzi mensili adjusted (total return, dividendi inclusi)
raw    = yf.download(ALL_TICKERS, start=START, end=END, auto_adjust=True, progress=True)
prices = raw['Close'].resample('ME').last()
prices.columns = [c if c != '^GSPC' else 'SP500' for c in prices.columns]
prices.dropna(how='all', inplace=True)

# Rendimenti logaritmici mensili: r_t = ln(P_t / P_{t-1})
returns = np.log(prices / prices.shift(1)).dropna()

# Serie di rendimento del mercato (proxy: SP500)
r_m = returns['SP500']                        # serie temporale del mercato

# Ticker "rischiosi" = tutti e 10 (SP500 ha beta=1 per definizione, utile come verifica)
RISKY = [c for c in returns.columns]          # tutti e 10 incluso SP500

print(f'Osservazioni: {len(returns)} mesi  ({returns.index[0].date()} → {returns.index[-1].date()})')
returns.head(3)

---
## Blocco 2 – Stima OLS dei beta (Punto 8)

### Modello di regressione

Il **Single Index Model** assume che il rendimento di ogni asset $i$ sia:

$$r_{i,t} = \alpha_i + \beta_i \cdot r_{m,t} + \varepsilon_{i,t}$$

dove:
- $\alpha_i$: **intercetta** (alpha di Jensen) — rendimento in eccesso rispetto a quanto previsto dal rischio sistematico
- $\beta_i$: **beta** — sensibilità al mercato; misura il rischio sistematico (non diversificabile)
- $\varepsilon_{i,t}$: errore idiosincratico, con $\mathbb{E}[\varepsilon_i]=0$ e $\text{Cov}(\varepsilon_i, r_m)=0$

### Stima OLS

L'estimatore OLS (Ordinary Least Squares) minimizza la somma dei quadrati dei residui:

$$\hat{\beta}_i = \frac{\text{Cov}(r_i, r_m)}{\text{Var}(r_m)}, \qquad \hat{\alpha}_i = \bar{r}_i - \hat{\beta}_i \cdot \bar{r}_m$$

### Errori standard e test di significatività

Con $T$ osservazioni e $\hat{\sigma}^2_\varepsilon = \text{RSS}/(T-2)$ (varianza residua con $T-2$ gradi di libertà):

$$\text{SE}(\hat{\beta}_i) = \sqrt{\frac{\hat{\sigma}^2_\varepsilon}{\sum_t(r_{m,t}-\bar{r}_m)^2}}, \qquad
\text{SE}(\hat{\alpha}_i) = \sqrt{\hat{\sigma}^2_\varepsilon \left(\frac{1}{T} + \frac{\bar{r}_m^2}{\sum_t(r_{m,t}-\bar{r}_m)^2}\right)}$$

Le statistiche $t = \hat{\cdot}/\text{SE}(\hat{\cdot})$ seguono una distribuzione $t$ con $T-2$ gradi di libertà.

**Interpretazione dell'alpha**: il CAPM prevede $\alpha_i = 0$ per tutti gli asset. Se $\alpha_i$ è statisticamente significativo, l'asset ha prodotto rendimenti superiori (α>0) o inferiori (α<0) alle previsioni del CAPM.

In [ ]:
# =============================================================================
# FUNZIONE OLS
# =============================================================================
# Implementazione manuale dell'OLS per avere pieno controllo sulle formule.
# Non si usa statsmodels per mantenere le dipendenze minime.

def ols_stats(y_series, x_series):
    """
    Stima OLS: y_t = alpha + beta * x_t + eps_t
    Restituisce un dizionario con tutti i parametri e le statistiche.
    """
    y = np.array(y_series, dtype=float)
    x = np.array(x_series, dtype=float)
    T = len(y)

    # ── Stime OLS ────────────────────────────────────────────────────────────
    # beta = Cov(y,x) / Var(x)  →  stima non distorta (ddof=1)
    beta  = np.cov(y, x, ddof=1)[0, 1] / np.var(x, ddof=1)
    alpha = y.mean() - beta * x.mean()     # alpha = ȳ - beta * x̄

    # ── Residui e varianza residua ────────────────────────────────────────────
    eps = y - alpha - beta * x             # residui OLS
    RSS = eps @ eps                        # Residual Sum of Squares
    s2  = RSS / (T - 2)                   # varianza residua con T-2 dof

    # ── Errori standard ───────────────────────────────────────────────────────
    Sxx    = ((x - x.mean())**2).sum()    # = (T-1) * Var(x, ddof=1)
    se_b   = np.sqrt(s2 / Sxx)
    se_a   = np.sqrt(s2 * (1/T + x.mean()**2 / Sxx))

    # ── t-statistiche e p-value (due code, dist. t con T-2 dof) ──────────────
    t_b  = beta  / se_b
    t_a  = alpha / se_a
    p_b  = 2 * stats.t.sf(abs(t_b), df=T - 2)
    p_a  = 2 * stats.t.sf(abs(t_a), df=T - 2)

    # ── R² = 1 - RSS/TSS ──────────────────────────────────────────────────────
    TSS = ((y - y.mean())**2).sum()
    R2  = 1 - RSS / TSS

    return dict(alpha=alpha, beta=beta,
                se_a=se_a, se_b=se_b,
                t_a=t_a,   t_b=t_b,
                p_a=p_a,   p_b=p_b,
                R2=R2, eps=eps, s2=s2)

# =============================================================================
# STIMA OLS PER TUTTI I 10 TITOLI
# =============================================================================
ols = {ticker: ols_stats(returns[ticker], r_m) for ticker in RISKY}

# ── Tabella riassuntiva ────────────────────────────────────────────────────────
# Simboli di significatività: *** p<0.01, ** p<0.05, * p<0.1
def sig_stars(p):
    if p < 0.01:  return '***'
    if p < 0.05:  return '**'
    if p < 0.10:  return '*'
    return ''

rows = []
for t in RISKY:
    o = ols[t]
    rows.append({
        'Ticker':        t,
        'Alpha (ann %)': round(o['alpha'] * ann * 100, 3),
        't(alpha)':      round(o['t_a'], 3),
        'Sig alpha':     sig_stars(o['p_a']),
        'Beta':          round(o['beta'], 4),
        't(beta)':       round(o['t_b'], 3),
        'Sig beta':      sig_stars(o['p_b']),
        'R²':            round(o['R2'], 4),
    })

tab = pd.DataFrame(rows).set_index('Ticker')
print('Stima OLS: r_i = alpha + beta * r_m + eps   (T =', len(returns), 'mesi)\n')
display(tab)
print('\nLegenda: *** p<0.01  ** p<0.05  * p<0.10')

---
## Blocco 3 – Visualizzazione dei beta

Il **beta** misura la sensibilità del rendimento dell'asset alle variazioni del mercato:
- β > 1: asset **aggressivo** — amplifica i movimenti del mercato
- β = 1: asset si muove in linea con il mercato (per costruzione: β(SP500) = 1)
- β < 1: asset **difensivo** — attutisce le variazioni del mercato
- β < 0: asset contro-ciclico — si muove in direzione opposta al mercato (raro)

In [ ]:
betas_all  = np.array([ols[t]['beta']  for t in RISKY])
alphas_all = np.array([ols[t]['alpha'] for t in RISKY])

# ── Bar chart dei beta ─────────────────────────────────────────────────────────
# Colori: asset con beta > 1 in rosso (aggressivi), beta < 1 in blu (difensivi)
colors_bar = ['tomato' if b > 1 else 'steelblue' for b in betas_all]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Beta
ax = axes[0]
bars = ax.bar(RISKY, betas_all, color=colors_bar, edgecolor='black', linewidth=0.5)
ax.axhline(1.0, color='black', lw=1.5, ls='--', label='β = 1 (mercato)')
ax.axhline(0.0, color='grey',  lw=0.8, ls=':')
for bar, b in zip(bars, betas_all):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{b:.2f}', ha='center', va='bottom', fontsize=8)
ax.set_title('Beta dei 10 titoli vs S&P 500', fontsize=12)
ax.set_ylabel('Beta (β)')
ax.legend(fontsize=9)
ax.tick_params(axis='x', rotation=30)
red_p  = mpatches.Patch(color='tomato',    label='β > 1 (aggressivo)')
blue_p = mpatches.Patch(color='steelblue', label='β < 1 (difensivo)')
ax.legend(handles=[red_p, blue_p], fontsize=9)

# Alpha annualizzato
ax = axes[1]
alpha_ann = alphas_all * ann * 100
colors_a  = ['limegreen' if a > 0 else 'salmon' for a in alpha_ann]
bars_a    = ax.bar(RISKY, alpha_ann, color=colors_a, edgecolor='black', linewidth=0.5)
ax.axhline(0, color='black', lw=1)
for bar, a in zip(bars_a, alpha_ann):
    ypos = bar.get_height() + 0.3 if a >= 0 else bar.get_height() - 1.5
    ax.text(bar.get_x() + bar.get_width()/2, ypos,
            f'{a:.1f}%', ha='center', va='bottom', fontsize=8)
ax.set_title('Alpha di Jensen annualizzato (%/anno)', fontsize=12)
ax.set_ylabel('Alpha annuo (%)')
ax.tick_params(axis='x', rotation=30)
g_p = mpatches.Patch(color='limegreen', label='α > 0 (outperformance)')
s_p = mpatches.Patch(color='salmon',    label='α < 0 (underperformance)')
ax.legend(handles=[g_p, s_p], fontsize=9)

fig.suptitle('Beta e Alpha di Jensen – Stima OLS (2015–2025)', fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig('beta_alpha_B8.png', dpi=150)
plt.show()
print('Grafico salvato: beta_alpha_B8.png')

---
## Blocco 4 – Security Market Line (SML)

La **Security Market Line** è la relazione del CAPM tra rendimento atteso e beta:

$$\mathbb{E}[r_i] = r_f + \beta_i \cdot (\mathbb{E}[r_m] - r_f)$$

È una retta nello spazio $(\beta, \mu)$ che passa per $(0, r_f)$ e $(1, \mu_m)$.

- Asset **sopra la SML**: alpha positivo → rendimento superiore alle previsioni CAPM (potenzialmente sottovalutato)
- Asset **sotto la SML**: alpha negativo → rendimento inferiore alle previsioni CAPM (potenzialmente sopravvalutato)

**Differenza SML vs CML**: la CML (vista nella Parte A) usa σ sull'asse x ed è valida solo per portafogli efficienti. La SML usa β sull'asse x ed è valida per qualsiasi asset (anche non efficienti).

In [ ]:
mu_m   = r_m.mean()                     # rendimento medio mensile del mercato
mu_emp = returns.mean()                 # rendimenti medi empirici mensili

# SML: E[r_i] = rf + beta_i * (mu_m - rf)  →  retta nello spazio (beta, mu)
beta_grid   = np.linspace(min(betas_all) - 0.1, max(betas_all) + 0.1, 200)
mu_sml      = rf + beta_grid * (mu_m - rf)      # SML mensile

# Rendimento CAPM previsto per ogni titolo (mensile)
mu_capm_all = rf + betas_all * (mu_m - rf)

fig, ax = plt.subplots(figsize=(11, 7))

# SML
ax.plot(beta_grid, mu_sml * ann * 100, 'b-', lw=2,
        label=f'SML: E[r] = rf + β·(μ_m − rf)  (rf={rf_annual*100:.1f}%, μ_m={mu_m*ann*100:.1f}%)')

# Punto mercato (beta=1, mu=mu_m)
ax.scatter(1, mu_m * ann * 100, color='black', s=200, marker='D', zorder=6,
           label='S&P 500 (mercato, β=1)')

# Singoli asset
sector_map = {}
palette    = {'Tech': 'steelblue', 'Healthcare': 'seagreen', 'Energy': 'tomato', 'Index': 'black'}
for sector, tickers in TICKERS.items():
    for t in tickers:
        name = 'SP500' if t == '^GSPC' else t
        sector_map[name] = sector

for i, ticker in enumerate(RISKY):
    mu_real = mu_emp[ticker] * ann * 100       # rendimento reale annualizzato
    mu_pred = mu_capm_all[i] * ann * 100       # rendimento CAPM previsto
    color   = palette[sector_map[ticker]]
    ax.scatter(betas_all[i], mu_real, color=color, s=120, zorder=5,
               edgecolors='black', linewidths=0.5)
    ax.annotate(ticker, (betas_all[i], mu_real),
                textcoords='offset points', xytext=(6, 3), fontsize=9)
    # Freccia verticale dal punto CAPM al punto reale (= alpha annualizzato)
    if abs(mu_real - mu_pred) > 1.5:    # mostra solo alpha rilevanti
        ax.annotate('', xy=(betas_all[i], mu_real), xytext=(betas_all[i], mu_pred),
                    arrowprops=dict(arrowstyle='->', color='grey', lw=1.2))

# rf sull'asse
ax.scatter(0, rf_annual * 100, color='grey', s=100, marker='o', zorder=5,
           label=f'rf = {rf_annual*100:.1f}%')

# Legenda colori settori
legend_patches = [mpatches.Patch(color=c, label=s) for s, c in palette.items()]
ax.legend(handles=legend_patches + ax.get_legend_handles_labels()[0][:2],
          fontsize=8, loc='upper left')

ax.set_xlabel('Beta (β)', fontsize=12)
ax.set_ylabel('Rendimento Atteso Annuo (%)', fontsize=12)
ax.set_title('Security Market Line (SML) – CAPM vs Rendimenti Reali', fontsize=13)
ax.axhline(0, color='grey', lw=0.5, ls=':')
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('SML_B8.png', dpi=150)
plt.show()
print('Grafico salvato: SML_B8.png')
print('\nAsset sopra la SML (alpha > 0): outperformance rispetto al CAPM')
print('Asset sotto la SML (alpha < 0): underperformance rispetto al CAPM')

---
## Blocco 5 – Matrice varianza-covarianza via Single Index Model (Punto 9)

Il **Single Index Model** semplifica la struttura della covarianza: invece di stimare le $\binom{N}{2}$ covarianze empiriche, le ricostruisce tramite solo $2N+1$ parametri (N beta, N varianze residue, 1 varianza di mercato):

$$\Sigma_{\text{SIM}}[i,i] = \beta_i^2 \sigma^2_m + \sigma^2_{\varepsilon_i} \quad \text{(varianza totale)}$$

$$\Sigma_{\text{SIM}}[i,j] = \beta_i \cdot \beta_j \cdot \sigma^2_m \quad (i \neq j) \quad \text{(tutta la correlazione passa dal mercato)}$$

In forma matriciale:
$$\Sigma_{\text{SIM}} = \boldsymbol{\beta}\boldsymbol{\beta}' \cdot \sigma^2_m + \mathbf{D}$$
dove $\boldsymbol{\beta} = (\beta_1, \ldots, \beta_N)'$ e $\mathbf{D} = \text{diag}(\sigma^2_{\varepsilon_1}, \ldots, \sigma^2_{\varepsilon_N})$.

**Interpretazione**: il SIM assume che l'unica fonte di correlazione tra i titoli sia il fattore comune (il mercato). Due titoli si muovono insieme **solo** perché entrambi rispondono al mercato.

In [ ]:
# ── Parametri del SIM per tutti i 10 titoli ────────────────────────────────────
betas_arr  = np.array([ols[t]['beta'] for t in RISKY])  # vettore beta (N,)
s2eps_arr  = np.array([ols[t]['s2']   for t in RISKY])  # varianze residue (N,)
sigma2_m   = np.var(r_m, ddof=1)                         # varianza mensile del mercato

# ── Matrice Sigma_SIM (N×N) ────────────────────────────────────────────────────
# np.outer(betas, betas) * sigma2_m  →  matrice N×N con Cov_SIM(i,j)=bi*bj*s2m
# np.diag(s2eps_arr)                 →  aggiunge le varianze idiosincratiche sulla diagonale
Sigma_SIM  = np.outer(betas_arr, betas_arr) * sigma2_m + np.diag(s2eps_arr)

# Confronto diagonale: Sigma_SIM[i,i] deve essere ≈ Var(r_i) empirica
var_empirical = np.array([returns[t].var(ddof=1) for t in RISKY])
var_sim       = np.diag(Sigma_SIM)

print('Verifica: varianza SIM vs varianza empirica (devono coincidere per costruzione):')
check = pd.DataFrame({'Var SIM (×10⁴)':    (var_sim       * 10000).round(4),
                      'Var empirica (×10⁴)': (var_empirical * 10000).round(4),
                      'Differenza':           ((var_sim - var_empirical) * 10000).round(6)},
                     index=RISKY)
display(check)

# ── Confronto matrici di correlazione: SIM vs empirica ────────────────────────
# Normalizziamo Sigma_SIM per ottenere la matrice di correlazione SIM
D_inv   = np.diag(1 / np.sqrt(var_sim))
Corr_SIM = D_inv @ Sigma_SIM @ D_inv
Corr_SIM_df = pd.DataFrame(Corr_SIM, index=RISKY, columns=RISKY)

Corr_EMP = returns.corr()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, corr, title in zip(axes,
                            [Corr_SIM_df, Corr_EMP],
                            ['Correlazione SIM (β·β\'·σ²_m)', 'Correlazione Empirica']):
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, vmin=-1, vmax=1, square=True,
                linewidths=0.5, ax=ax, annot_kws={'size': 8})
    ax.set_title(title, fontsize=12)
fig.suptitle('Confronto matrici di correlazione: SIM vs Empirica', fontsize=13)
fig.tight_layout()
fig.savefig('correlation_SIM_vs_empirical_B9.png', dpi=150)
plt.show()
print('Salvato: correlation_SIM_vs_empirical_B9.png')
print('\nNota: il SIM tende ad appiattire le correlazioni perché assume')
print('che tutta la co-movimentazione passi esclusivamente dal mercato.')

---
## Blocco 6 – Rendimenti attesi CAPM (Punto 9)

Il **CAPM** impone che $\alpha_i = 0$ per ogni asset in equilibrio. Il rendimento atteso è quindi **interamente** spiegato dalla compensazione per il rischio sistematico:

$$\mathbb{E}[r_i]_{\text{CAPM}} = r_f + \beta_i \cdot (\mu_m - r_f)$$

Confrontiamo i rendimenti CAPM con quelli empirici: la differenza è l'**alpha di Jensen** stimato nell'OLS. Se il CAPM fosse perfettamente valido, la differenza sarebbe zero per tutti gli asset.

In [ ]:
# ── Rendimenti attesi CAPM mensili ─────────────────────────────────────────────
# E[r_i]_CAPM = rf + beta_i * (mu_m - rf)
mu_capm_arr  = rf + betas_arr * (mu_m - rf)       # (N,) mensili
mu_emp_arr   = np.array([returns[t].mean() for t in RISKY])

# ── Tabella confronto ─────────────────────────────────────────────────────────
comp = pd.DataFrame({
    'Beta':                 betas_arr.round(4),
    'μ CAPM annuo (%)':     (mu_capm_arr * ann * 100).round(2),
    'μ Empirico annuo (%)': (mu_emp_arr  * ann * 100).round(2),
    'Alpha annuo (%)':      ((mu_emp_arr - mu_capm_arr) * ann * 100).round(2),
    'p-value alpha':        [round(ols[t]['p_a'], 4) for t in RISKY],
    'Alpha sign.':          [sig_stars(ols[t]['p_a']) for t in RISKY],
}, index=RISKY)

print('Confronto rendimenti empirici vs CAPM:\n')
display(comp)

# ── Grafico scatter: mu_empirico vs mu_CAPM ────────────────────────────────────
# Se il CAPM fosse esatto, tutti i punti starebbero sulla bisettrice y=x
fig, ax = plt.subplots(figsize=(8, 8))
all_mu = np.concatenate([mu_capm_arr, mu_emp_arr]) * ann * 100
lim    = (min(all_mu) - 2, max(all_mu) + 2)
ax.plot(lim, lim, 'k--', lw=1, label='y = x (CAPM perfetto)')
for i, t in enumerate(RISKY):
    color = palette[sector_map[t]]
    ax.scatter(mu_capm_arr[i]*ann*100, mu_emp_arr[i]*ann*100,
               color=color, s=120, zorder=4, edgecolors='black', lw=0.5)
    ax.annotate(t, (mu_capm_arr[i]*ann*100, mu_emp_arr[i]*ann*100),
                textcoords='offset points', xytext=(5, 3), fontsize=9)
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel('Rendimento CAPM annuo (%)'); ax.set_ylabel('Rendimento Empirico annuo (%)')
ax.set_title('CAPM previsto vs Rendimento reale\n(punti sopra la diagonale = alpha > 0)', fontsize=12)
legend_patches = [mpatches.Patch(color=c, label=s) for s, c in palette.items()]
ax.legend(handles=legend_patches, fontsize=9)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('capm_vs_empirical_B9.png', dpi=150)
plt.show()
print('Salvato: capm_vs_empirical_B9.png')

---
## Blocco 7 – Frontiera efficiente SIM vs Empirica (Punto 9)

Costruiamo la frontiera efficiente usando:
- **μ_CAPM** come vettore dei rendimenti attesi (invece di μ empirico)
- **Σ_SIM** come matrice di varianza-covarianza (invece di Σ empirica)

per i **5 titoli selezionati nella Parte A** (NVDA, MSFT, JNJ, MRK, XOM).

Il confronto mostra se la semplificazione del SIM/CAPM produce una frontiera simile a quella empirica.
In genere il SIM tende a **comprimere** la frontiera perché:
1. Le correlazioni SIM sono spesso più basse di quelle empiriche (meno diversificazione apparente)
2. I rendimenti attesi CAPM possono differire dai rendimenti storici (alpha ≠ 0 nel campione)

In [ ]:
# ── Estrai i parametri SIM e CAPM per i 5 titoli selezionati ──────────────────
sel_idx     = [RISKY.index(t) for t in SELECTED]   # indici nel vettore RISKY

mu5_capm    = mu_capm_arr[sel_idx]                  # mu CAPM (5,) mensili
mu5_emp     = mu_emp_arr[sel_idx]                   # mu empirico (5,) mensili
Sig5_SIM    = Sigma_SIM[np.ix_(sel_idx, sel_idx)]   # Sigma SIM (5×5)
Sig5_emp    = returns[SELECTED].cov().values         # Sigma empirica (5×5)

n5    = len(SELECTED)
iota5 = np.ones(n5)

def markowitz_frontier(mu_vec, Sig_mat, label=''):
    """
    Calcola la frontiera efficiente analitica (Markowitz) data μ e Σ.
    Restituisce gli array (sigma_annualizzata, mu_annualizzata) della frontiera.
    """
    Sinv  = np.linalg.inv(Sig_mat)
    A_mk  = float(iota5 @ Sinv @ iota5)
    B_mk  = float(iota5 @ Sinv @ mu_vec)
    C_mk  = float(mu_vec @ Sinv @ mu_vec)
    D_mk  = A_mk * C_mk - B_mk**2
    mu_gmv = B_mk / A_mk
    mu_grid = np.linspace(mu_gmv, mu_vec.max() * 1.6, 300)
    sig_grid = np.sqrt((A_mk * mu_grid**2 - 2*B_mk*mu_grid + C_mk) / D_mk)
    return sig_grid * np.sqrt(ann) * 100, mu_grid * ann * 100, mu_gmv, A_mk, B_mk, C_mk, D_mk

# ── Frontiera empirica (usa mu e Sigma empirici, come nella Parte A) ───────────
sig_emp_front, mu_emp_front, mu_gmv_emp, *_ = markowitz_frontier(mu5_emp, Sig5_emp, 'Empirica')

# ── Frontiera SIM (usa mu_CAPM e Sigma_SIM) ────────────────────────────────────
sig_sim_front, mu_sim_front, mu_gmv_sim, *_ = markowitz_frontier(mu5_capm, Sig5_SIM, 'SIM/CAPM')

# ── Grafico confronto ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(sig_emp_front, mu_emp_front, 'b-',  lw=2.5,
        label='Frontiera empirica (μ empirico, Σ empirica) – Parte A')
ax.plot(sig_sim_front, mu_sim_front, 'r--', lw=2.5,
        label='Frontiera SIM/CAPM (μ_CAPM, Σ_SIM) – Parte B')

# Singoli asset: punto empirico (cerchio) vs punto CAPM (croce)
for i, t in enumerate(SELECTED):
    sig_a = returns[t].std() * np.sqrt(ann) * 100
    mu_a  = returns[t].mean() * ann * 100
    mu_c  = mu5_capm[i] * ann * 100
    ax.scatter(sig_a, mu_a,  s=100, marker='o', color='steelblue',
               zorder=5, edgecolors='black', lw=0.5)
    ax.scatter(sig_a, mu_c,  s=100, marker='x', color='red',
               zorder=5, linewidths=1.5)
    ax.annotate(t, (sig_a, mu_a), textcoords='offset points', xytext=(5, 3), fontsize=9)

# Legenda extra per i marker dei singoli asset
from matplotlib.lines import Line2D
extra = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='steelblue',
           markersize=9, markeredgecolor='black', label='Asset: μ empirico'),
    Line2D([0],[0], marker='x', color='red', markersize=9,
           lw=0, markeredgewidth=2, label='Asset: μ CAPM'),
]
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles=handles + extra, fontsize=9)

ax.set_xlabel('Deviazione Standard Annua (%)'); ax.set_ylabel('Rendimento Atteso Annuo (%)')
ax.set_title('Frontiera Efficiente: Empirica (Parte A) vs SIM/CAPM (Parte B)\n5 titoli selezionati – short selling ammesso', fontsize=13)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('frontier_SIM_vs_empirical_B9.png', dpi=150)
plt.show()
print('Salvato: frontier_SIM_vs_empirical_B9.png')
print('\nCommento:')
print('- La frontiera SIM differisce da quella empirica principalmente per i')
print('  rendimenti attesi: il CAPM prevede μ_i = rf + β_i*(μ_m - rf),')
print('  che può differire molto dai μ empirici quando gli alpha sono grandi.')
print('- La struttura delle covarianze SIM è più parsimoniona (2N+1 parametri')
print('  invece di N(N+1)/2), ma può perdere informazione sulle correlazioni')
print('  idiosincratiche tra settori diversi.')